# The Transformer by hand: encoder and decoder

This is the full homework for **Attention by Hand**. You'll push one short sentence through a whole encoder layer, then translate it with a whole decoder layer, and **write down every matrix yourself**: from the token IDs to the probabilities of the next German word.

**How it works**

1. Run the first code cell. It sets up the sentence, a tiny vocabulary and the encoder's weights. Part 2 has its own setup cell for the decoder.
2. Each step shows a code cell full of `nan`. Replace every `nan` with your number, then run the cell. It checks your answer straight away.
3. If something is off, the checker says which entries and gives a hint for the first one. Still stuck? Call `reveal('name')` to see the answer and carry on.

Work on paper or use Python as a calculator for single numbers (`np.sin(1)`, `np.exp(0.5)`, `0.3*2 + 1.1*0.5`). Try not to use `@` to multiply whole matrices: doing the dot products yourself is the point. Two decimals is enough. Each check uses *your* previous checked matrices, so small rounding differences won't snowball.

**The sizes we use** (the same idea as the site, just small enough to do by hand)

| setting | here | paper |
|---|---|---|
| source length n | 3 ("the cat sat") | any |
| decoder positions m | 4 (⟨s⟩ die katze saß) | any |
| d_model | 4 | 512 |
| heads h | 2 | 8 |
| d_k = d_model / h | 2 | 64 |
| d_ff | 8 | 2048 |
| vocabulary V | 11 | ≈ 37,000 |
| layers N | 1 encoder + 1 decoder | 6 + 6 |

Dropout is skipped, as it would be when the model is used after training.

---

# Part 1: the encoder

In [ ]:
# Run this cell first. It sets up the example, the weights and the checker.
# You don't need to read it, and please don't peek at REF: that's the answer key.
import numpy as np
np.set_printoptions(precision=4, suppress=True)
nan = np.nan

# ---- settings for this run (paper values in comments) ----
sentence = "the cat sat"
n       = 3    # tokens            (any length)
d_model = 4    # vector width      (paper: 512)
h       = 2    # heads             (paper: 8)
d_k     = d_model // h   # = 2     (paper: 64)
d_ff    = 8    # FFN inner width   (paper: 2048)

# ---- a tiny vocabulary and its (made-up) learned embedding table, V x d_model ----
vocab = {"<pad>": 0, "the": 1, "cat": 2, "sat": 3, "on": 4, "mat": 5}
W_emb = np.array([
    [ 0.0,  0.0,  0.0,  0.0],   # 0 <pad>
    [ 0.1,  0.2, -0.1,  0.3],   # 1 the
    [ 0.4, -0.2,  0.3,  0.1],   # 2 cat
    [-0.3,  0.5,  0.2, -0.1],   # 3 sat
    [ 0.2,  0.1, -0.4,  0.2],   # 4 on
    [ 0.3, -0.1,  0.1,  0.4],   # 5 mat
])

# ---- attention weights (made up; a trained model learns these), each d_model x d_k ----
W_Q1 = np.array([[ 1.0, 0.0], [0.0,  1.0], [ 0.5, 0.0], [0.0, -0.5]])
W_K1 = np.array([[ 0.0, 1.0], [1.0,  0.0], [ 0.0, 0.5], [-0.5, 0.0]])
W_V1 = np.array([[ 1.0, 0.0], [0.0,  0.0], [ 0.0, 1.0], [ 0.5, 0.5]])
W_Q2 = np.array([[ 0.0, 0.5], [1.0,  0.0], [ 0.0, 1.0], [ 0.5, 0.0]])
W_K2 = np.array([[ 0.5, 0.0], [0.0,  1.0], [ 1.0, 0.0], [ 0.0, 0.5]])
W_V2 = np.array([[ 0.0, 1.0], [0.5,  0.0], [ 1.0, 0.0], [ 0.0, -0.5]])
W_O  = np.array([[ 0.5, 0.0, 0.0, 0.5],
                 [ 0.0, 0.5, 0.5, 0.0],
                 [ 0.0, 0.5, -0.5, 0.0],
                 [ 0.5, 0.0, 0.0, -0.5]])

# ---- feed-forward weights: W1 is d_model x d_ff, W2 is d_ff x d_model ----
W1 = np.array([[ 0.5, -0.5,  0.0,  1.0,  0.0, -1.0,  0.5,  0.0],
               [ 0.0,  0.5,  0.5,  0.0, -0.5,  0.0,  0.0,  1.0],
               [ 1.0,  0.0, -0.5,  0.0,  0.5,  0.5,  0.0, -0.5],
               [ 0.0,  0.0,  0.5, -0.5,  0.0,  0.0,  1.0,  0.0]])
b1 = np.array([0.0, 0.1, 0.0, -0.1, 0.0, 0.1, 0.0, -0.1])
W2 = np.array([[ 0.5,  0.0,  0.0,  0.0],
               [ 0.0,  0.5,  0.0,  0.0],
               [ 0.0,  0.0,  0.5,  0.0],
               [ 0.0,  0.0,  0.0,  0.5],
               [-0.5,  0.0,  0.5,  0.0],
               [ 0.0, -0.5,  0.0,  0.5],
               [ 0.5,  0.5,  0.0,  0.0],
               [ 0.0,  0.0, -0.5, -0.5]])
b2 = np.array([0.1, 0.0, -0.1, 0.0])
eps = 1e-5   # LayerNorm's tiny safety constant; it won't change your 2-decimal answers

# =============== the checker (no need to read below) ===============
def _sm(S):
    e = np.exp(S - S.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
def _ln(R):
    mu = R.mean(axis=1); sd = np.sqrt(R.var(axis=1) + eps)
    return mu, sd, (R - mu[:, None]) / sd[:, None]

_ok = {}
def got(k): return _ok[k] if k in _ok else REF[k]

_RULES = {   # how each matrix is computed from the matrices before it
  "ids":  lambda: np.array([vocab[t] for t in sentence.split()], dtype=float),
  "E":    lambda: W_emb[got("ids").astype(int)],
  "Es":   lambda: got("E") * np.sqrt(d_model),
  "div":  lambda: np.array([10000 ** (2 * i / d_model) for i in range(d_model // 2)]),
  "PE":   lambda: np.array([[f(p / got("div")[c // 2]) for c, f in
                             zip(range(d_model), [np.sin, np.cos] * (d_model // 2))] for p in range(n)]),
  "X":    lambda: got("Es") + got("PE"),
  "Q1":   lambda: got("X") @ W_Q1, "K1": lambda: got("X") @ W_K1, "V1": lambda: got("X") @ W_V1,
  "Q2":   lambda: got("X") @ W_Q2, "K2": lambda: got("X") @ W_K2, "V2": lambda: got("X") @ W_V2,
  "S1":   lambda: got("Q1") @ got("K1").T, "S2": lambda: got("Q2") @ got("K2").T,
  "S1s":  lambda: got("S1") / np.sqrt(d_k), "S2s": lambda: got("S2") / np.sqrt(d_k),
  "A1":   lambda: _sm(got("S1s")), "A2": lambda: _sm(got("S2s")),
  "H1":   lambda: got("A1") @ got("V1"), "H2": lambda: got("A2") @ got("V2"),
  "C":    lambda: np.concatenate([got("H1"), got("H2")], axis=1),
  "MHA":  lambda: got("C") @ W_O,
  "R1":   lambda: got("X") + got("MHA"),
  "mu1":  lambda: _ln(got("R1"))[0], "sd1": lambda: _ln(got("R1"))[1],
  "X1":   lambda: (got("R1") - got("mu1")[:, None]) / got("sd1")[:, None],
  "P":    lambda: got("X1") @ W1 + b1,
  "Hf":   lambda: np.maximum(0, got("P")),
  "F":    lambda: got("Hf") @ W2 + b2,
  "R2":   lambda: got("X1") + got("F"),
  "mu2":  lambda: _ln(got("R2"))[0], "sd2": lambda: _ln(got("R2"))[1],
  "Z":    lambda: (got("R2") - got("mu2")[:, None]) / got("sd2")[:, None],
}
REF = {}
for _k, _f in _RULES.items():   # the full answer key, computed in order
    REF[_k] = _f()

_HINTS = {
  "ids": "Look each word up in the vocab dictionary.",
  "E": "Row r is the row of W_emb whose number is ids[r]. No multiplication, just copy the row.",
  "Es": "Multiply every entry of E by sqrt(d_model) = sqrt(4) = 2.",
  "div": "Pair i uses 10000 ** (2*i / d_model). With d_model = 4 that's 10000**0 and 10000**0.5.",
  "PE": "Even column 2i: sin(pos / div[i]). Odd column 2i+1: cos(pos / div[i]). Angles are in radians.",
  "X": "Add Es and PE entry by entry (same row, same column).",
  "S1s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "S2s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "A1": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "A2": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "C": "Put H1 and H2 side by side: the first d_k columns are H1, the next d_k are H2.",
  "R1": "Residual: add the layer's input X to MHA, entry by entry.",
  "mu1": "The mean of each row of R1 (sum the 4 numbers, divide by 4).",
  "sd1": "For each row: average the squared distances from the row's mean, then take the square root.",
  "X1": "(R1[r, c] - mu1[r]) / sd1[r]. Every row ends up with mean 0 and spread 1.",
  "P": "X1 @ W1, then add b1[c] to every entry in column c.",
  "Hf": "ReLU: keep positive numbers, replace negatives with 0.",
  "F": "Hf @ W2, then add b2[c] to every entry in column c.",
  "R2": "Residual again: X1 + F, entry by entry.",
  "mu2": "The mean of each row of R2.",
  "sd2": "The standard deviation of each row of R2 (same recipe as sd1).",
  "Z": "(R2[r, c] - mu2[r]) / sd2[r].",
}
_MATMUL = {"Q1": ("X", W_Q1, "W_Q1"), "K1": ("X", W_K1, "W_K1"), "V1": ("X", W_V1, "W_V1"),
           "Q2": ("X", W_Q2, "W_Q2"), "K2": ("X", W_K2, "W_K2"), "V2": ("X", W_V2, "W_V2"),
           "S1": ("Q1", None, "K1ᵀ"), "S2": ("Q2", None, "K2ᵀ"),
           "H1": ("A1", None, "V1"), "H2": ("A2", None, "V2"), "MHA": ("C", W_O, "W_O")}

def _hint(name, r, c):
    if name in _MATMUL:
        a, B, bname = _MATMUL[name]
        A = got(a)
        if B is None:
            B = {"S1": lambda: got("K1").T, "S2": lambda: got("K2").T,
                 "H1": lambda: got("V1"), "H2": lambda: got("V2")}[name]()
        terms = " + ".join(f"({A[r, j]:.2f})×({B[j, c]:.2f})" for j in range(A.shape[1]))
        return f"{name}[{r},{c}] = row {r} of {a} · column {c} of {bname} = {terms}"
    return _HINTS.get(name, "")

def check(name, yours):
    """Grade your matrix. Wrong or blank cells are listed with a hint for the first one."""
    exp = _RULES[name]()
    y = np.asarray(yours, dtype=float)
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    tol = 0.02 + 0.01 * np.abs(exp)
    bad = np.argwhere(np.isnan(y) | (np.abs(y - exp) > tol))
    if len(bad) == 0:
        _ok[name] = y
        print(f"✓ {name} {exp.shape} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        where = f"[{', '.join(map(str, idx))}]"
        print(f"   {name}{where}: you have {y[tuple(idx)]:.4g}")
    idx = bad[0]
    r, c = (int(idx[0]), int(idx[1])) if y.ndim == 2 else (int(idx[0]), 0)
    print("   Hint:", _hint(name, r, c))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def reveal(name):
    """Show the answer for one matrix and use it from here on."""
    exp = _RULES[name](); _ok[name] = exp
    print(f"{name} {exp.shape} =\n{np.round(exp, 4)}")

def check_shapes(answers):
    """Grade a dictionary of shapes for the paper's sizes."""
    PAPER = {"E": (6, 512), "W_Q1": (512, 64), "Q1": (6, 64), "S1": (6, 6), "A1": (6, 6),
             "H1": (6, 64), "C": (6, 512), "W_O": (512, 512), "P": (6, 2048), "Z": (6, 512)}
    right = 0
    for k, v in PAPER.items():
        mine = tuple(answers.get(k, ()))
        ok = mine == v; right += ok
        print(f"{'✓' if ok else '✗'} {k:5s} you said {mine or '(blank)'}" + ("" if ok else f", the paper has {v}"))
    print(f"\n{right} of {len(PAPER)} right.")

print("Ready. Sentence:", sentence.split(), "| n =", n, "| d_model =", d_model, "| h =", h, "| d_k =", d_k, "| d_ff =", d_ff)

## Step 1: Tokenize

The sentence is `"the cat sat"`. Look each word up in `vocab` to get its ID.

**Shape:** (n,) = (3,)

In [ ]:
# one ID per token
ids = np.array([nan, nan, nan])
check('ids', ids)

## Step 2: Embedding lookup

`W_emb` has one row per vocabulary entry, so it's (6 × 4) here and (≈37,000 × 512) in the paper. The lookup copies out the row for each ID.

**Shape:** (n × d_model) = (3 × 4)

In [ ]:
print(W_emb)

In [ ]:
# row r = W_emb[ids[r]]
E = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('E', E)

## Step 3: Scale by √d_model

The paper multiplies the embeddings by √d_model (§3.4). Here that's √4 = 2. In the paper it's √512 ≈ 22.63.

**Shape:** (3 × 4), unchanged

In [ ]:
# every entry of E times 2
Es = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Es', Es)

## Step 4: Positional encoding

First the divisor for each pair, `10000 ** (2i / d_model)`. With d_model = 4 there are 2 pairs.

In [ ]:
# div[0] for pair 0, div[1] for pair 1
div = np.array([nan, nan])
check('div', div)

Now the PE matrix. For position `pos` and pair `i`:

- column 2i: `sin(pos / div[i])`
- column 2i+1: `cos(pos / div[i])`

Positions are 0, 1, 2. Use radians, and give 4 decimals for this one.

**Shape:** (n × d_model) = (3 × 4)

In [ ]:
# row = position 0, 1, 2; columns: sin, cos (pair 0), sin, cos (pair 1)
PE = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('PE', PE)

## Step 5: Add them: X = Es + PE

This X is what enters the encoder layer.

**Shape:** (3 × 4)

In [ ]:
# entry by entry
X = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('X', X)

## Step 6: Queries, keys and values for each head

Each head has its own W_Q, W_K and W_V, each (d_model × d_k) = (4 × 2). Multiply X by each one:

(3 × 4) · (4 × 2) = (3 × 2)

Entry [r, c] is row r of X dotted with column c of the weight matrix.

In [ ]:
print('W_Q1 =\n', W_Q1, '\nW_K1 =\n', W_K1, '\nW_V1 =\n', W_V1)

### Head 1

In [ ]:
# X · W_Q1
Q1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('Q1', Q1)

In [ ]:
# X · W_K1
K1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('K1', K1)

In [ ]:
# X · W_V1
V1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('V1', V1)

### Head 2

In [ ]:
print('W_Q2 =\n', W_Q2, '\nW_K2 =\n', W_K2, '\nW_V2 =\n', W_V2)

In [ ]:
# X · W_Q2
Q2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('Q2', Q2)

In [ ]:
# X · W_K2
K2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('K2', K2)

In [ ]:
# X · W_V2
V2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('V2', V2)

## Step 7: Attention scores S = Q · Kᵀ

Every token's query against every token's key: (3 × 2) · (2 × 3) = (3 × 3). Entry [r, c] says how well token r's query matches token c's key.

In [ ]:
# head 1: S1[r, c] = Q1[r] · K1[c]
S1 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('S1', S1)

In [ ]:
# head 2
S2 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('S2', S2)

## Step 8: Scale by √d_k

Divide every score by √d_k = √2 ≈ 1.4142 (in the paper, √64 = 8). This keeps the softmax from getting too peaky.

In [ ]:
# S1 / sqrt(2)
S1s = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('S1s', S1s)

In [ ]:
# S2 / sqrt(2)
S2s = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('S2s', S2s)

## Step 9: Softmax, row by row

For each row: `e**x` for every entry, then divide by the row's total. Every row must add up to 1. Row r is how token r spreads its attention over the 3 tokens.

**Shape:** (3 × 3)

In [ ]:
# head 1 attention weights
A1 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('A1', A1)

In [ ]:
# head 2 attention weights
A2 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('A2', A2)

## Step 10: Weighted sum of values, head = A · V

(3 × 3) · (3 × 2) = (3 × 2). Each output row is a blend of the value rows, mixed using that token's attention weights.

In [ ]:
# A1 · V1
H1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('H1', H1)

In [ ]:
# A2 · V2
H2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('H2', H2)

## Step 11: Concatenate the heads and mix with W_O

Put the heads side by side: h × (3 × 2) gives (3 × 4). Then multiply by W_O (4 × 4).

In [ ]:
# [H1 | H2]
C = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('C', C)

In [ ]:
print(W_O)

In [ ]:
# C · W_O
MHA = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('MHA', MHA)

## Step 12: Add & Norm (1)

**Residual:** add the input back in, `R1 = X + MHA`.

**LayerNorm:** for each row on its own, subtract the row's mean and divide by its standard deviation. Here γ = 1 and β = 0, so there's no extra scale or shift.

In [ ]:
# X + MHA
R1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R1', R1)

In [ ]:
# mean of each row of R1
mu1 = np.array([nan, nan, nan])
check('mu1', mu1)

In [ ]:
# standard deviation of each row of R1 (divide by 4, not 3)
sd1 = np.array([nan, nan, nan])
check('sd1', sd1)

In [ ]:
# (R1[r, c] - mu1[r]) / sd1[r]
X1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('X1', X1)

## Step 13: Feed-forward network

Expand with W1 (4 × 8) plus b1, apply ReLU, then compress with W2 (8 × 4) plus b2. Each token goes through on its own, so there's no mixing between rows here.

(3 × 4) · (4 × 8) = (3 × 8), then (3 × 8) · (8 × 4) = (3 × 4)

In [ ]:
print('W1 =\n', W1, '\nb1 =', b1)

In [ ]:
# X1 · W1 + b1
P = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('P', P)

In [ ]:
# ReLU(P)
Hf = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('Hf', Hf)

In [ ]:
print('W2 =\n', W2, '\nb2 =', b2)

In [ ]:
# Hf · W2 + b2
F = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('F', F)

## Step 14: Add & Norm (2), the encoder output

Same recipe as step 12, with X1 as the residual.

In [ ]:
# X1 + F
R2 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R2', R2)

In [ ]:
# mean of each row of R2
mu2 = np.array([nan, nan, nan])
check('mu2', mu2)

In [ ]:
# standard deviation of each row of R2
sd2 = np.array([nan, nan, nan])
check('sd2', sd2)

In [ ]:
# (R2[r, c] - mu2[r]) / sd2[r]
Z = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Z', Z)

## Encoder done

Z is the output of one encoder layer: one 4-number vector per English token, mixed with context from the whole sentence. In the paper you'd repeat the layer 6 times. Here Z goes straight to the decoder.

Run the cell below to see which encoder matrices you did yourself.

In [ ]:
for k in ["ids", "E", "Es", "PE", "X", "Q1", "K1", "V1", "S1", "A1", "H1", "C", "MHA", "X1", "P", "F", "Z"]:
    print(f"{k:4s} {str(got(k).shape):8s} {'✓ yours' if k in _ok else '· answer key'}")
print(f"\n{len(_ok)} of {len(_RULES)} matrices done by you.")

---

# Part 2: the decoder

The decoder is a **next-word predictor**. At every position it answers one question: *given the English sentence and the German words so far, what is the next German word?* Everything new in this part follows from that job:

- it reads the German sentence **shifted right**, starting with a start token ⟨s⟩,
- its self-attention is **masked**, so no position can peek at the word it's supposed to predict,
- its **cross-attention** takes queries from the German side and keys and values from the encoder output Z,
- it ends with a **linear layer and softmax** that turn each row into probabilities over the vocabulary.

We do the training-time pass: the whole (correct) translation goes in at once, and all 4 predictions come out together.

Run the setup cell first.

In [ ]:
# Run this cell before the decoder part. It adds the German words, the decoder's
# weights and the decoder's answer key to the checker. Again, no peeking at REF.

# ---- the shared vocabulary grows: the paper uses ONE vocabulary for both languages (§3.4) ----
vocab.update({"<s>": 6, "</s>": 7, "die": 8, "katze": 9, "saß": 10})
if len(W_emb) == 6:   # (so rerunning this cell is safe)
  W_emb = np.vstack([W_emb, np.array([
    [ 0.2,  0.3, -0.4, -0.3],   # 6 <s>   start token
    [-0.1,  0.1,  0.3, -0.2],   # 7 </s>  end token
    [-0.4, -0.3, -0.1, -0.1],   # 8 die
    [ 0.4, -0.1,  0.2,  0.1],   # 9 katze
    [-0.4,  0.2, -0.4, -0.4],   # 10 saß
  ])])
words = {i: w for w, i in vocab.items()}
V = len(vocab)   # 11 here, about 37,000 in the paper

# ---- the translation, shifted right ----
target    = "die katze saß"
dec_input = ["<s>", "die", "katze", "saß"]     # what the decoder reads
dec_label = ["die", "katze", "saß", "</s>"]    # what it must predict at each position
m = len(dec_input)                            # 4 positions (the source has n = 3)
inf = np.inf

# ---- masked self-attention weights (the decoder's own; not shared with the encoder) ----
W_mQ1 = np.array([[ 0.0, -0.5], [ 0.0,  0.0], [-1.0, -1.0], [ 0.5,  1.0]])
W_mK1 = np.array([[ 0.5,  0.5], [ 0.0, -1.0], [ 0.0,  0.0], [ 0.5,  0.0]])
W_mV1 = np.array([[ 0.0, -1.0], [ 0.5, -0.5], [ 0.0, -1.0], [ 0.0,  0.0]])
W_mQ2 = np.array([[ 0.0,  0.0], [-1.0,  0.0], [ 0.5,  1.0], [-0.5,  0.5]])
W_mK2 = np.array([[ 0.0,  0.0], [ 0.5, -0.5], [-0.5,  0.0], [ 0.0,  1.0]])
W_mV2 = np.array([[-0.5,  0.0], [ 1.0,  1.0], [ 0.5,  0.0], [ 0.5, -0.5]])
W_mO  = np.array([[ 0.0, -0.5,  0.0, -0.5],
                  [ 0.5, -0.5,  0.0,  0.0],
                  [ 0.5,  0.0, -0.5,  0.0],
                  [-0.5,  0.5,  0.5,  0.0]])

# ---- cross-attention weights: W_xQ is applied to the decoder, W_xK and W_xV to Z ----
W_xQ1 = np.array([[-1.0,  0.0], [-0.5,  0.0], [-0.5,  0.0], [ 0.5, -0.5]])
W_xK1 = np.array([[ 0.0,  0.0], [-0.5,  0.0], [ 0.5,  0.0], [ 0.5,  0.0]])
W_xV1 = np.array([[ 1.0, -1.0], [ 0.0,  1.0], [ 0.0, -1.0], [ 1.0, -0.5]])
W_xQ2 = np.array([[-1.0, -0.5], [ 1.0,  0.0], [-1.0,  0.0], [-1.0,  0.0]])
W_xK2 = np.array([[ 0.0,  0.5], [-1.0,  0.0], [ 0.0,  1.0], [-1.0,  0.0]])
W_xV2 = np.array([[ 0.0,  0.0], [ 0.5,  0.0], [ 0.5, -1.0], [-0.5, -0.5]])
W_xO  = np.array([[ 0.0,  0.0, -0.5,  0.0],
                  [ 0.5, -0.5,  0.0, -0.5],
                  [ 0.5,  0.0,  0.5,  0.5],
                  [ 0.5,  0.0,  0.0,  0.0]])

# ---- the decoder's feed-forward network ----
W1d = np.array([[-0.5,  0.5, -0.5,  0.0,  0.5, -0.5, -0.5,  0.0],
                [-1.0,  0.0, -0.5, -1.0, -1.0, -0.5,  0.0,  0.0],
                [-0.5, -1.0,  0.0,  0.0,  0.0,  0.0, -0.5, -1.0],
                [ 0.0, -0.5, -1.0,  1.0,  0.0,  0.0, -0.5, -0.5]])
b1d = np.array([0.0, 0.1, -0.1, 0.0, -0.1, 0.0, 0.1, 0.1])
W2d = np.array([[ 0.0,  0.0,  0.0, -0.5],
                [-0.5,  0.0, -0.5,  0.5],
                [ 0.5,  0.0,  0.5,  0.0],
                [ 0.0,  0.5,  0.0, -0.5],
                [ 0.0,  0.5,  0.0,  0.0],
                [ 0.0,  0.0,  0.5,  0.5],
                [ 0.0,  0.5,  0.5, -0.5],
                [ 0.0,  0.0,  0.0,  0.0]])
b2d = np.array([-0.1, 0.1, 0.0, -0.1])

MASK = np.triu(np.full((m, m), -np.inf), 1)    # used by the checker only; you build it yourself

# =============== the decoder checker (no need to read below) ===============
_DEC = {
  "ids_d": lambda: np.array([vocab[t] for t in dec_input], dtype=float),
  "Ed":    lambda: W_emb[got("ids_d").astype(int)],
  "Eds":   lambda: got("Ed") * np.sqrt(d_model),
  "PEd":   lambda: np.array([[f(p / got("div")[c // 2]) for c, f in
                              zip(range(d_model), [np.sin, np.cos] * (d_model // 2))] for p in range(m)]),
  "Y0":    lambda: got("Eds") + got("PEd"),
}
for _h in "12":
    _DEC.update({
      f"mQ{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mQ{h}"])(_h),
      f"mK{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mK{h}"])(_h),
      f"mV{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mV{h}"])(_h),
      f"mS{_h}": (lambda h: lambda: got(f"mQ{h}") @ got(f"mK{h}").T)(_h),
      f"mS{_h}s": (lambda h: lambda: got(f"mS{h}") / np.sqrt(d_k))(_h),
      f"mS{_h}m": (lambda h: lambda: got(f"mS{h}s") + MASK)(_h),
      f"mA{_h}": (lambda h: lambda: _sm(got(f"mS{h}m")))(_h),
      f"mH{_h}": (lambda h: lambda: got(f"mA{h}") @ got(f"mV{h}"))(_h),
    })
_DEC.update({
  "mC":   lambda: np.concatenate([got("mH1"), got("mH2")], axis=1),
  "mMHA": lambda: got("mC") @ W_mO,
  "R1d":  lambda: got("Y0") + got("mMHA"),
  "mu1d": lambda: _ln(got("R1d"))[0], "sd1d": lambda: _ln(got("R1d"))[1],
  "Y1":   lambda: (got("R1d") - got("mu1d")[:, None]) / got("sd1d")[:, None],
})
for _h in "12":
    _DEC.update({
      f"xQ{_h}": (lambda h: lambda: got("Y1") @ globals()[f"W_xQ{h}"])(_h),
      f"xK{_h}": (lambda h: lambda: got("Z") @ globals()[f"W_xK{h}"])(_h),
      f"xV{_h}": (lambda h: lambda: got("Z") @ globals()[f"W_xV{h}"])(_h),
      f"xS{_h}": (lambda h: lambda: got(f"xQ{h}") @ got(f"xK{h}").T)(_h),
      f"xS{_h}s": (lambda h: lambda: got(f"xS{h}") / np.sqrt(d_k))(_h),
      f"xA{_h}": (lambda h: lambda: _sm(got(f"xS{h}s")))(_h),
      f"xH{_h}": (lambda h: lambda: got(f"xA{h}") @ got(f"xV{h}"))(_h),
    })
_DEC.update({
  "xC":    lambda: np.concatenate([got("xH1"), got("xH2")], axis=1),
  "xMHA":  lambda: got("xC") @ W_xO,
  "R2d":   lambda: got("Y1") + got("xMHA"),
  "mu2d":  lambda: _ln(got("R2d"))[0], "sd2d": lambda: _ln(got("R2d"))[1],
  "Y2":    lambda: (got("R2d") - got("mu2d")[:, None]) / got("sd2d")[:, None],
  "Pd":    lambda: got("Y2") @ W1d + b1d,
  "Hd":    lambda: np.maximum(0, got("Pd")),
  "Fd":    lambda: got("Hd") @ W2d + b2d,
  "R3d":   lambda: got("Y2") + got("Fd"),
  "mu3d":  lambda: _ln(got("R3d"))[0], "sd3d": lambda: _ln(got("R3d"))[1],
  "Yout":  lambda: (got("R3d") - got("mu3d")[:, None]) / got("sd3d")[:, None],
  "logits": lambda: got("Yout") @ W_emb.T,
  "probs": lambda: _sm(got("logits")),
  "pred":  lambda: got("probs").argmax(axis=1).astype(float),
  "nll":   lambda: -np.log(got("probs")[np.arange(m), [vocab[w] for w in dec_label]]),
})
_RULES.update(_DEC)
for _k, _f in _DEC.items():
    REF[_k] = _f()

_HINTS.update({
  "ids_d": "Look up each decoder INPUT token (⟨s⟩ first), not the words it has to predict.",
  "Ed": "Same lookup as the encoder: row r = W_emb[ids_d[r]].",
  "Eds": "Multiply every entry by sqrt(d_model) = 2, exactly like the encoder.",
  "PEd": "Same formula as the encoder's PE. Rows 0 to 2 are identical to PE; only row 3 (position 3) is new.",
  "Y0": "Eds + PEd, entry by entry.",
  "mS1s": "Divide every score by sqrt(2) ≈ 1.4142.", "mS2s": "Divide every score by sqrt(2) ≈ 1.4142.",
  "mS1m": "Keep entries with column ≤ row. Every entry ABOVE the diagonal (column > row) becomes -inf.",
  "mS2m": "Keep entries with column ≤ row. Every entry ABOVE the diagonal (column > row) becomes -inf.",
  "mA1": "Softmax each row. e**(-inf) = 0, so masked entries get exactly 0 weight; divide by the total of the rest.",
  "mA2": "Softmax each row. e**(-inf) = 0, so masked entries get exactly 0 weight; divide by the total of the rest.",
  "mC": "Put mH1 and mH2 side by side.",
  "R1d": "Residual: Y0 + mMHA.", "mu1d": "Mean of each row of R1d.", "sd1d": "Standard deviation of each row of R1d (divide by 4).",
  "Y1": "(R1d[r, c] - mu1d[r]) / sd1d[r].",
  "xS1s": "Divide every score by sqrt(2).", "xS2s": "Divide every score by sqrt(2).",
  "xA1": "Softmax each row over the 3 ENGLISH words. No mask: the whole source sentence is allowed.",
  "xA2": "Softmax each row over the 3 ENGLISH words. No mask: the whole source sentence is allowed.",
  "xC": "Put xH1 and xH2 side by side.",
  "R2d": "Residual: Y1 + xMHA.", "mu2d": "Mean of each row of R2d.", "sd2d": "Standard deviation of each row of R2d.",
  "Y2": "(R2d[r, c] - mu2d[r]) / sd2d[r].",
  "Pd": "Y2 @ W1d, then add b1d[c] to column c.", "Hd": "ReLU: negatives become 0.",
  "Fd": "Hd @ W2d, then add b2d[c] to column c.",
  "R3d": "Residual: Y2 + Fd.", "mu3d": "Mean of each row of R3d.", "sd3d": "Standard deviation of each row of R3d.",
  "Yout": "(R3d[r, c] - mu3d[r]) / sd3d[r].",
  "logits": "logits[r, w] = Yout[r] · W_emb[w]: the decoder's vector dotted with word w's embedding row.",
  "probs": "Softmax each row of logits over all V words.",
  "pred": "For each row of probs, the ID (column number) of the biggest probability.",
  "nll": "nll[r] = -ln(probs[r, id of dec_label[r]]): minus the log of the probability given to the CORRECT next word.",
})
_DEC_MM = {}
for _h in "12":
    _DEC_MM.update({
      f"mQ{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mQ{h}"])(_h), f"W_mQ{_h}"),
      f"mK{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mK{h}"])(_h), f"W_mK{_h}"),
      f"mV{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mV{h}"])(_h), f"W_mV{_h}"),
      f"mS{_h}": (f"mQ{_h}", (lambda h: lambda: got(f"mK{h}").T)(_h), f"mK{_h}ᵀ"),
      f"mH{_h}": (f"mA{_h}", (lambda h: lambda: got(f"mV{h}"))(_h), f"mV{_h}"),
      f"xQ{_h}": ("Y1", (lambda h: lambda: globals()[f"W_xQ{h}"])(_h), f"W_xQ{_h}"),
      f"xK{_h}": ("Z", (lambda h: lambda: globals()[f"W_xK{h}"])(_h), f"W_xK{_h}"),
      f"xV{_h}": ("Z", (lambda h: lambda: globals()[f"W_xV{h}"])(_h), f"W_xV{_h}"),
      f"xS{_h}": (f"xQ{_h}", (lambda h: lambda: got(f"xK{h}").T)(_h), f"xK{_h}ᵀ"),
      f"xH{_h}": (f"xA{_h}", (lambda h: lambda: got(f"xV{h}"))(_h), f"xV{_h}"),
    })
_DEC_MM.update({"mMHA": ("mC", lambda: W_mO, "W_mO"), "xMHA": ("xC", lambda: W_xO, "W_xO"),
                "logits": ("Yout", lambda: W_emb.T, "W_embᵀ")})
_enc_hint = globals().get('_enc_hint', _hint)
def _hint(name, r, c):
    if name in _DEC_MM:
        a, Bf, bname = _DEC_MM[name]
        A, B = got(a), Bf()
        terms = " + ".join(f"({A[r, j]:.2f})×({B[j, c]:.2f})" for j in range(A.shape[1]))
        return f"{name}[{r},{c}] = row {r} of {a} · column {c} of {bname} = {terms}"
    return _enc_hint(name, r, c)

def check(name, yours):
    """Grade your matrix. Wrong or blank cells are listed with a hint for the first one."""
    exp = _RULES[name]()
    y = np.asarray(yours, dtype=float)
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    fin = np.isfinite(exp)
    tol = 0.02 + 0.01 * np.abs(np.where(fin, exp, 0))
    with np.errstate(invalid="ignore"):
        good = np.where(fin, np.abs(y - np.where(fin, exp, 0)) <= tol, y == exp)
    if name == "pred":
        good = y == exp
    bad = np.argwhere(np.isnan(y) | ~good)
    if len(bad) == 0:
        _ok[name] = y
        print(f"✓ {name} {exp.shape} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        where = f"[{', '.join(map(str, idx))}]"
        print(f"   {name}{where}: you have {y[tuple(idx)]:.4g}")
    idx = bad[0]
    r, c = (int(idx[0]), int(idx[1])) if y.ndim == 2 else (int(idx[0]), 0)
    print("   Hint:", _hint(name, r, c))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def check_dec_shapes(answers):
    """Grade a dictionary of decoder shapes for the paper's sizes (6 English tokens, 7 decoder positions)."""
    PAPER = {"Y0": (7, 512), "mS1": (7, 7), "mA1": (7, 7), "xQ1": (7, 64), "xK1": (6, 64),
             "xS1": (7, 6), "xH1": (7, 64), "Pd": (7, 2048), "Yout": (7, 512), "logits": (7, 37000)}
    right = 0
    for k, v in PAPER.items():
        mine = tuple(answers.get(k, ()))
        ok = mine == v; right += ok
        print(f"{'✓' if ok else '✗'} {k:6s} you said {mine or '(blank)'}" + ("" if ok else f", the paper has {v}"))
    print(f"\n{right} of {len(PAPER)} right.")

print("Decoder ready. Source:", sentence.split(), "→ decoder input:", dec_input, "| m =", m, "| V =", V)

## Step 15: Shift the target right and tokenize

The translation of "the cat sat" is "die katze saß". The decoder reads it **shifted one place to the right**, behind a start token:

| position | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| decoder input | ⟨s⟩ | die | katze | saß |
| must predict | die | katze | saß | ⟨/s⟩ |

Position 2 has seen `⟨s⟩ die katze` and must predict `saß`. Look up the IDs of the **decoder input** in `vocab` (run `print(vocab)` if you like).

**Shape:** (m,) = (4,)

In [ ]:
print(vocab)

In [ ]:
# one ID per decoder input token, ⟨s⟩ first
ids_d = np.array([nan, nan, nan, nan])
check('ids_d', ids_d)

## Step 16: Embed, scale and add positions

Exactly the encoder's steps 2 to 5, now for the 4 German input tokens. The embedding table is the **same** shared `W_emb` (now 11 rows), and the positional encoding uses the same formula. Rows 0 to 2 of `PEd` are identical to the encoder's `PE`, because position 1 means the same thing in both stacks.

**Shape:** (m × d_model) = (4 × 4)

In [ ]:
# row r = W_emb[ids_d[r]]
Ed = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Ed', Ed)

In [ ]:
# Ed × sqrt(4) = Ed × 2
Eds = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Eds', Eds)

In [ ]:
# positions 0, 1, 2, 3; 4 decimals
PEd = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('PEd', PEd)

In [ ]:
# Eds + PEd: the input to decoder layer 1
Y0 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Y0', Y0)

## Step 17: Masked self-attention, queries, keys and values

The first sublayer works like encoder self-attention: Q, K and V **all come from Y0**. The decoder has its own weights (`W_mQ1` and so on).

(4 × 4) · (4 × 2) = (4 × 2)

In [ ]:
print('W_mQ1 =\n', W_mQ1, '\nW_mK1 =\n', W_mK1, '\nW_mV1 =\n', W_mV1)

### Head 1

In [ ]:
# Y0 · W_mQ1
mQ1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mQ1', mQ1)

In [ ]:
# Y0 · W_mK1
mK1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mK1', mK1)

In [ ]:
# Y0 · W_mV1
mV1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mV1', mV1)

### Head 2

In [ ]:
print('W_mQ2 =\n', W_mQ2, '\nW_mK2 =\n', W_mK2, '\nW_mV2 =\n', W_mV2)

In [ ]:
# Y0 · W_mQ2
mQ2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mQ2', mQ2)

In [ ]:
# Y0 · W_mK2
mK2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mK2', mK2)

In [ ]:
# Y0 · W_mV2
mV2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mV2', mV2)

## Step 18: Scores, then scale

Every German position against every German position: (4 × 2) · (2 × 4) = (4 × 4). Then divide by √d_k = √2.

In [ ]:
# mQ1 · mK1ᵀ
mS1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS1', mS1)

In [ ]:
# mS1 / sqrt(2)
mS1s = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS1s', mS1s)

In [ ]:
# mQ2 · mK2ᵀ
mS2 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS2', mS2)

In [ ]:
# mS2 / sqrt(2)
mS2s = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS2s', mS2s)

## Step 19: The mask

This is the one new idea in decoder self-attention. Row r may only look at columns 0 to r. Every entry **above the diagonal** (column > row) is replaced with **−∞** (§3.2.3). Type it as `-inf`.

```
row ⟨s⟩    [ ok   -inf -inf -inf ]
row die    [ ok   ok   -inf -inf ]
row katze  [ ok   ok   ok   -inf ]
row saß    [ ok   ok   ok   ok   ]
```

Why? Position 2 must *predict* "saß", and "saß" is sitting right there at input position 3. Without the mask, the model would just copy it.

In [ ]:
# mS1s with -inf above the diagonal
mS1m = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS1m', mS1m)

In [ ]:
# mS2s with -inf above the diagonal
mS2m = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mS2m', mS2m)

## Step 20: Softmax with the mask

Softmax each row as before. Since e<sup>−∞</sup> = 0, every masked entry gets **exactly 0**, and the remaining entries of the row share the whole weight. Row ⟨s⟩ can only see itself, so it's `[1, 0, 0, 0]`.

In [ ]:
# head 1; zeros above the diagonal
mA1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mA1', mA1)

In [ ]:
# head 2
mA2 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mA2', mA2)

## Step 21: Blend the values, join the heads, Add & Norm

`mH = mA · mV` (4 × 2). The zeros in mA mean that row r only mixes values from positions 0 to r. Then concatenate the heads, multiply by `W_mO`, add the residual and normalize each row, exactly like the encoder.

In [ ]:
# mA1 · mV1
mH1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mH1', mH1)

In [ ]:
# mA2 · mV2
mH2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('mH2', mH2)

In [ ]:
# [mH1 | mH2]
mC = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mC', mC)

In [ ]:
print(W_mO)

In [ ]:
# mC · W_mO
mMHA = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('mMHA', mMHA)

In [ ]:
# Y0 + mMHA
R1d = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R1d', R1d)

In [ ]:
# mean of each row of R1d
mu1d = np.array([nan, nan, nan, nan])
check('mu1d', mu1d)

In [ ]:
# standard deviation of each row of R1d
sd1d = np.array([nan, nan, nan, nan])
check('sd1d', sd1d)

In [ ]:
# (R1d - mu1d) / sd1d, row by row
Y1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Y1', Y1)

## Step 22: Cross-attention, where Q, K and V come from

This is the bridge between the two halves (§3.2.3, "encoder-decoder attention"). The arithmetic is the same as before. What changes is **where the inputs come from**:

- **Q from the decoder:** `xQ = Y1 · W_xQ` → (4 × 2), one query per German position
- **K and V from the encoder:** `xK = Z · W_xK`, `xV = Z · W_xV` → (3 × 2), one per English word

The question comes from the German side ("which English words matter for my next word?"). What is searched, and what is handed back, comes from the English side.

In [ ]:
print('Z (from Part 1) =\n', np.round(got('Z'), 4))
print('W_xQ1 =\n', W_xQ1, '\nW_xK1 =\n', W_xK1, '\nW_xV1 =\n', W_xV1)

### Head 1

In [ ]:
# Y1 · W_xQ1   (decoder side)
xQ1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xQ1', xQ1)

In [ ]:
# Z · W_xK1    (encoder side: 3 rows!)
xK1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xK1', xK1)

In [ ]:
# Z · W_xV1    (encoder side)
xV1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xV1', xV1)

### Head 2

In [ ]:
print('W_xQ2 =\n', W_xQ2, '\nW_xK2 =\n', W_xK2, '\nW_xV2 =\n', W_xV2)

In [ ]:
# Y1 · W_xQ2
xQ2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xQ2', xQ2)

In [ ]:
# Z · W_xK2
xK2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xK2', xK2)

In [ ]:
# Z · W_xV2
xV2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xV2', xV2)

## Step 23: Cross-attention scores and softmax

(4 × 2) · (2 × 3) = **(4 × 3)**. The score table is no longer square: rows are German positions, columns are English words. Scale by √2, then softmax each row. **No mask here:** the whole English sentence is known from the start.

In [ ]:
# xQ1 · xK1ᵀ
xS1 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xS1', xS1)

In [ ]:
# xS1 / sqrt(2)
xS1s = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xS1s', xS1s)

In [ ]:
# softmax of each row; each row adds up to 1
xA1 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xA1', xA1)

In [ ]:
# xQ2 · xK2ᵀ
xS2 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xS2', xS2)

In [ ]:
# xS2 / sqrt(2)
xS2s = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xS2s', xS2s)

In [ ]:
# softmax of each row
xA2 = np.array([
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan],
    [nan, nan, nan]
])
check('xA2', xA2)

## Step 24: Blend the English values, join the heads, Add & Norm

(4 × 3) · (3 × 2) = (4 × 2). Each German position now pulls in a mix of the English words' information. The output has 4 rows because there are 4 **queries**.

In [ ]:
# xA1 · xV1
xH1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xH1', xH1)

In [ ]:
# xA2 · xV2
xH2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('xH2', xH2)

In [ ]:
# [xH1 | xH2]
xC = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('xC', xC)

In [ ]:
print(W_xO)

In [ ]:
# xC · W_xO
xMHA = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('xMHA', xMHA)

In [ ]:
# Y1 + xMHA
R2d = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R2d', R2d)

In [ ]:
# mean of each row of R2d
mu2d = np.array([nan, nan, nan, nan])
check('mu2d', mu2d)

In [ ]:
# standard deviation of each row of R2d
sd2d = np.array([nan, nan, nan, nan])
check('sd2d', sd2d)

In [ ]:
# (R2d - mu2d) / sd2d
Y2 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Y2', Y2)

## Step 25: Feed-forward network and the last Add & Norm

Same recipe as the encoder's FFN, with the decoder's own weights. Each position goes through on its own.

In [ ]:
print('W1d =\n', W1d, '\nb1d =', b1d)

In [ ]:
# Y2 · W1d + b1d
Pd = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('Pd', Pd)

In [ ]:
# ReLU(Pd)
Hd = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('Hd', Hd)

In [ ]:
print('W2d =\n', W2d, '\nb2d =', b2d)

In [ ]:
# Hd · W2d + b2d
Fd = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Fd', Fd)

In [ ]:
# Y2 + Fd
R3d = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R3d', R3d)

In [ ]:
# mean of each row of R3d
mu3d = np.array([nan, nan, nan, nan])
check('mu3d', mu3d)

In [ ]:
# standard deviation of each row of R3d
sd3d = np.array([nan, nan, nan, nan])
check('sd3d', sd3d)

In [ ]:
# (R3d - mu3d) / sd3d: the decoder's output
Yout = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('Yout', Yout)

## Step 26: Linear layer: a score for every word

The decoder turns each row into one score (a *logit*) per vocabulary word. The paper reuses the embedding table for this, transposed (§3.4, "weight tying"):

`logits = Yout · W_embᵀ` → (4 × 4) · (4 × 11) = (4 × 11)

Entry [r, w] is the dot product of position r's vector with word w's embedding: "how much does what I'm saying look like word w?" For this one step it's fine to use Python for each single dot product, like `Yout[0] @ W_emb[8]`, since there are 44 of them.

In [ ]:
print(W_emb)
print(words)

In [ ]:
# row = decoder position, column = word ID 0…10
logits = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
])
check('logits', logits)

## Step 27: Softmax: probabilities for the next word

Softmax each row over all 11 words. Row r is the model's guess for the word at target position r.

Then read off the **prediction** (the ID with the highest probability) and the **loss** for each position: minus the log of the probability the model gave the correct next word. Training lowers this number.

The weights are made up, so don't expect the right translation: a trained model would put most of the probability on "die", "katze", "saß" and ⟨/s⟩.

In [ ]:
# softmax of each row of logits
probs = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
])
check('probs', probs)

In [ ]:
# the word ID with the highest probability in each row
pred = np.array([nan, nan, nan, nan])
check('pred', pred)

In [ ]:
# -ln(probs[r, correct next word's ID])
nll = np.array([nan, nan, nan, nan])
check('nll', nll)

In [ ]:
print("position  input   must predict   model's guess   P(correct)   loss")
for r in range(m):
    p = got("probs")[r]; t = vocab[dec_label[r]]
    print(f"{r:8d}  {dec_input[r]:6s}  {dec_label[r]:13s}  {words[int(p.argmax())]:14s}  {p[t]:10.3f}  {-np.log(p[t]):5.2f}")
print(f"\nAverage loss over the {m} positions: {got('nll').mean():.3f}   (cross-entropy, §5.4; the paper also uses label smoothing 0.1)")

## You did it

You translated "the cat sat" through a whole Transformer by hand: one encoder layer, one decoder layer, and the output softmax. All 4 predictions came out of **one** pass, because the mask let every position train at once (this is *teacher forcing*).

At inference time there's no German sentence to feed in. The encoder runs once, then the decoder runs again for every new word: start with `[⟨s⟩]`, pick the most likely word, append it, and repeat until it picks ⟨/s⟩.

Run the cell below to see how much of the pass you did yourself.

In [ ]:
keys = list(_RULES)
mine = [k for k in keys if k in _ok]
print(f"{len(mine)} of {len(keys)} matrices done by you.")
print("still from the answer key:", [k for k in keys if k not in _ok] or "none!")

## Bonus: the paper's sizes, by hand

No numbers, just shapes. Imagine a 6-token English sentence, a German translation that makes the decoder input 7 positions long (⟨s⟩ + 6 words), and the paper's sizes (d_model 512, h 8, d_k 64, d_ff 2048, V ≈ 37,000). Fill in each shape as a tuple.

In [ ]:
paper_shapes = {"E": (), "W_Q1": (), "Q1": (), "S1": (), "A1": (), "H1": (), "C": (), "W_O": (), "P": (), "Z": ()}
check_shapes(paper_shapes)      # the encoder's shapes (6 tokens)

In [ ]:
dec_shapes = {
    "Y0":     (),   # decoder input after embedding + PE
    "mS1":    (),   # one head's masked self-attention scores
    "mA1":    (),   # its attention weights
    "xQ1":    (),   # cross-attention queries (one head)
    "xK1":    (),   # cross-attention keys (one head)
    "xS1":    (),   # cross-attention scores
    "xH1":    (),   # cross-attention head output
    "Pd":     (),   # FFN hidden layer
    "Yout":   (),   # decoder output
    "logits": (),   # one score per vocabulary word
}
check_dec_shapes(dec_shapes)

## Want more?

- **Predict one word at a time.** In the decoder setup cell, change `dec_input` to `["<s>", "die"]` and `dec_label` to `["die", "katze"]`, rerun it, and redo Part 2 with 2 rows (the `nan` cells need 2 rows too). Compare with your 4-row answers: the rows for ⟨s⟩ and "die" come out exactly the same, because the mask stops them from seeing later words. That's why real models cache them while generating.
- **Make the weights your own.** Edit any weight matrix and rerun. The checker recomputes the answer key for you.
- **Compare with the site.** Every step here matches a step in Attention by Hand, which also animates each multiplication.